# Task 3 — Turn Text Into Numbers (One-Hot Encoding)

**The problem in one sentence:** a machine-learning model does *math* — it can only work with numbers. But three of our columns contain **text**: `protocol_type` (`tcp`/`udp`/`icmp`), `service` (`http`/`smtp`/…), and `flag` (`SF`/`S0`/…). The model can't multiply `"tcp"`. So before training, we must convert that text into numbers — without inventing fake relationships. That conversion is this task.

**Terms defined as we go** (also in `reference/CONCEPTS_GLOSSARY.md`):
- **Categorical feature** = a column whose values are *labels/categories*, not quantities. `protocol_type` is categorical: `tcp`, `udp`, `icmp` are names, not amounts.
- **Numeric feature** = a column of actual numbers you can do math on, like `src_bytes`.
- **One-hot encoding** = the standard way to turn a categorical column into numbers (defined in detail in Step 2).

## Step 0 — Reload the data
Fresh notebook, so rebuild `df` like before: import pandas, define `all_columns`, `read_csv`, add `is_attack` as a **0/1 int** (remember `.astype(int)` this time!).

In [2]:
# TODO: reload the training data into df, with column names and a 0/1 is_attack column
import pandas as pd

feature_columns=[
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
]

all_columns=feature_columns+["label","difficulty"]

df=pd.read_csv("/Users/rohanb/06_projects/CYBERSECURITY/data/raw/KDDTrain+.txt", header=None, names=all_columns)
df

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.06,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20
125969,8,udp,private,SF,105,145,0,0,0,0,...,0.96,0.01,0.01,0.00,0.00,0.00,0.00,0.00,normal,21
125970,0,tcp,smtp,SF,2231,384,0,0,0,0,...,0.12,0.06,0.00,0.00,0.72,0.00,0.01,0.00,normal,18
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.03,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20


In [3]:
df["is_attack"]=(df["label"]!="normal").astype(int)
df

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty,is_attack
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20,0
1,0,udp,other,SF,146,0,0,0,0,0,...,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15,0
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19,1
3,0,tcp,http,SF,232,8153,0,0,0,0,...,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21,0
4,0,tcp,http,SF,199,420,0,0,0,0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.06,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20,1
125969,8,udp,private,SF,105,145,0,0,0,0,...,0.01,0.01,0.00,0.00,0.00,0.00,0.00,normal,21,0
125970,0,tcp,smtp,SF,2231,384,0,0,0,0,...,0.06,0.00,0.00,0.72,0.00,0.01,0.00,normal,18,0
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20,1


## Step 1 — See the problem for yourself
Let's confirm which columns are text vs numbers.

**Term — `dtype`:** short for 'data type'. pandas labels each column with the *kind* of data it holds: `int64`/`float64` = numbers; text shows as `object` (or `str` in newer pandas) — either way it means **text**.

**Your job:** print `df.dtypes` and find the columns that are text. Then, to build intuition, print the unique values of `protocol_type`.

**Hints:**
- `df.dtypes` lists every column and its type.
- `df["protocol_type"].unique()` shows the distinct values it contains.
- You'll find **4** text columns: `protocol_type`, `service`, `flag` — the 3 categorical *features* we'll encode — **plus `label`**, which is text too but is the *answer* (we drop it later in Step 5, we never encode it). Only the 3 features get one-hot encoded.

In [4]:
# TODO: print df.dtypes, and df["protocol_type"].unique()
df.dtypes

duration                         int64
protocol_type                      str
service                            str
flag                               str
src_bytes                        int64
dst_bytes                        int64
land                             int64
wrong_fragment                   int64
urgent                           int64
hot                              int64
num_failed_logins                int64
logged_in                        int64
num_compromised                  int64
root_shell                       int64
su_attempted                     int64
num_root                         int64
num_file_creations               int64
num_shells                       int64
num_access_files                 int64
num_outbound_cmds                int64
is_host_login                    int64
is_guest_login                   int64
count                            int64
srv_count                        int64
serror_rate                    float64
srv_serror_rate          

In [5]:
df["protocol_type"].unique()

<StringArray>
['tcp', 'udp', 'icmp']
Length: 3, dtype: str

In [6]:
df["service"].unique()

<StringArray>
[   'ftp_data',       'other',     'private',        'http',  'remote_job',
        'name',  'netbios_ns',       'eco_i',         'mtp',      'telnet',
      'finger',    'domain_u',      'supdup',   'uucp_path',      'Z39_50',
        'smtp',    'csnet_ns',        'uucp', 'netbios_dgm',       'urp_i',
        'auth',      'domain',         'ftp',         'bgp',        'ldap',
       'ecr_i',      'gopher',       'vmnet',      'systat',    'http_443',
         'efs',       'whois',       'imap4',    'iso_tsap',        'echo',
      'klogin',        'link',      'sunrpc',       'login',      'kshell',
     'sql_net',        'time',   'hostnames',        'exec',       'ntp_u',
     'discard',        'nntp',     'courier',         'ctf',         'ssh',
     'daytime',       'shell',     'netstat',       'pop_3',        'nnsp',
         'IRC',       'pop_2',     'printer',       'tim_i',     'pm_dump',
       'red_i', 'netbios_ssn',         'rje',         'X11',       'urh_i'

In [7]:
df["flag"].unique()

<StringArray>
['SF', 'S0', 'REJ', 'RSTR', 'SH', 'RSTO', 'S1', 'RSTOS0', 'S3', 'S2', 'OTH']
Length: 11, dtype: str

## Step 2 — Understand one-hot encoding (read this before coding)

**Why not just map tcp→1, udp→2, icmp→3?** Because that invents a *fake order/relationship*: it would tell the model `icmp (3) > udp (2) > tcp (1)`, and that `udp` is 'between' the other two. That's nonsense — they're just names, not ranks. A model would wrongly do math on those numbers.

**One-hot encoding** fixes this. It replaces one categorical column with **several new 0/1 columns — one per category** — where exactly one is 'hot' (=1) per row. Example for `protocol_type`:

| original | → | protocol_type_tcp | protocol_type_udp | protocol_type_icmp |
|---|---|---|---|---|
| tcp  | → | **1** | 0 | 0 |
| udp  | → | 0 | **1** | 0 |
| icmp | → | 0 | 0 | **1** |

Now each category is a separate yes/no (1/0) column with no fake ordering — pure numbers the model can safely use. ('One-hot' = one column is 'hot'/1, the rest are 0.)

Because `service` has ~70 possible values, one-hotting it creates ~70 columns — that's why our column count will jump a lot. That's expected and fine.

## Step 3 — Apply one-hot encoding
pandas has a built-in for this: **`pd.get_dummies`** ('dummy variables' is the statistics name for these 0/1 columns).

**Your job:** create a new DataFrame `df_encoded` where the 3 text columns are one-hot encoded, but all the numeric columns stay as they are.

**Hints:**
- `pd.get_dummies(df, columns=[...])` one-hot-encodes *only* the columns you list and leaves the rest untouched.
- Pass the 3 categorical column names in that `columns=` list.
- Assign the result to `df_encoded`.
- Optional: add `dtype=int` so the new columns are `0/1` ints instead of `True/False`.

In [8]:
# TODO: df_encoded = pd.get_dummies(df, columns=[...])
df_encoded=pd.get_dummies(df, columns=["protocol_type","service","flag"],dtype=int)
df_encoded

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0,491,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,0,146,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,0,232,8153,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
4,0,199,420,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
125969,8,105,145,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
125970,0,2231,384,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
125971,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


## Step 4 — Confirm the transformation
Check what changed.

**Your job:** compare `df.shape` (before) with `df_encoded.shape` (after), and peek at the new columns.

**Hints:**
- Print both shapes. Columns should jump from 44 to well over 100 (each category became its own column).
- `df_encoded.columns.tolist()` (or `df_encoded.head()`) lets you eyeball the new `protocol_type_tcp`, `service_http`, `flag_SF`, ... columns.
- Sanity check: `df_encoded["protocol_type_tcp"].sum()` should equal the number of tcp rows you saw back in Task 2 (102,689).

In [9]:
# TODO: print df.shape and df_encoded.shape; inspect the new columns
df.shape

(125973, 44)

In [10]:
df_encoded.shape

(125973, 125)

In [11]:
df_encoded["protocol_type_tcp"].sum()

np.int64(102689)

## Step 5 — Separate the clues from the answer (build X and y)
One more prep step. In ML we conventionally split the data into two pieces:
- **`X`** = the **features** (all the input clues the model looks at). Capital X by convention.
- **`y`** = the **target/label** (the single answer we want it to predict). Lowercase y by convention.

The model learns the mapping `X → y`. Crucially, `X` must NOT contain the answer or anything that gives it away, or the model 'cheats'.

**Your job:**
1. Make `y = df_encoded["is_attack"]`.
2. Make `X` = everything EXCEPT the columns that are the answer or leak it: drop `is_attack`, `label` (the attack name — that's the answer in words!), and `difficulty` (not a real network feature).

**Hints:**
- Drop columns with `df_encoded.drop(columns=[...])`.
- Wait — is `label` still a column after Step 3? You didn't one-hot it, so yes, it's still there as text. It must be dropped from `X` (it literally *is* the answer).
- Print `X.shape` and `y.shape`. `X` should have the same number of rows as `y`, and a few fewer columns than `df_encoded`.

In [ ]:
# TODO: build y (the target) and X (the features, with answer-columns dropped)
X= df_encoded.drop(columns=["is_attack", "label","difficulty"])
y=df_encoded["is_attack"]
print(X.shape)
print(y.shape)

(125973, 122)
(125973,)


### ✅ You pass Task 3 when:
1. You found the 3 text (`object`) columns and can say *why* text is a problem for a model.
2. `df_encoded` has 100+ columns, all numeric (no more `object` dtype among the features).
3. You can explain, in one sentence, why one-hot beats mapping tcp→1/udp→2/icmp→3.
4. `X` (features) and `y` (target) exist, `X` has no `label`/`is_attack`/`difficulty`, and `X.shape[0] == y.shape[0]`.

Paste me your shapes + your one-sentence answer to #3, and I'll review — then Task 4 is training the model on `X` and `y`.